In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.dbutils import *

from datetime import datetime, date

In [0]:
bronze_layer = "bronze"
silver_layer = "silver"

target_schema = "IPL_database"
bronze_schema = "bronze"
silver_schema = "silver"

In [0]:
if spark.catalog.tableExists("ipl_database.silver.batsman_stats"):
    df = spark.table(f"{target_schema}.{bronze_schema}.batsman_raw").filter(
        to_date(col("updated_at")) >= date.today()
    )
else:
    df = spark.table(f"{target_schema}.{bronze_schema}.batsman_raw")

In [0]:
%skip
df.columns

In [0]:
%skip
df = df.select(
    col("id").cast("string"),
    col("batsman").cast("string"),
    col("wicket_by").cast("string"),
    col("stats").cast("string"),
    col("batting_team").cast("string"),
    col("bowling_team").cast("string"),
    col("date").cast("string"),
    col("venue").cast("string"),
    col("winner").cast("string"),
    col("toss").cast("string"),
    col("runs_scored").cast("string"),
    col("_rescued_data").cast("string"),
    col("file_name").cast("string"),
    col("ingestion_time").cast("timestamp")
)

## 01 Cleaning batsman name column

In [0]:
%skip
df.select("batsman").distinct().display()

In [0]:
df = df.withColumn(
    "is_captain",
    when(col("batsman").contains("(c)"),lit(1)).otherwise(lit(0))
) \
.withColumn(
    "player_name",
    regexp_replace(col("batsman"),r'\(c\)',"")
) \
.withColumn(
    "player_name",
    regexp_replace(col("player_name"),"�","")
) \
.withColumn(
    "player_name",
    regexp_replace(col("player_name")," †","")
) \
.withColumn(
    "player_name",
    regexp_replace(col("player_name")," ","")
) \
.withColumn(
    "wicket_by",
    regexp_replace(col("wicket_by"),"�","")
) \
.withColumn(
    "wicket_by",
    regexp_replace(col("wicket_by")," ","")
)
# df.display(10)

## 02 Deal with stats column

In [0]:
df = df.withColumn(
  "stats_array",
  from_json(
    col("stats"),
    ArrayType(StringType())
  )
)
# df.display()

### Create new columns from stats column

mapping is 

- 0 : balls played by batsman
- 1 : minutes played
- 2 : 4's
- 3 : 6's
- 4 : String rate

In [0]:
def safe_convert(x):
    try:
        return int(x)
    except:
        return 0
    
# register a UDF
safe_udf = udf(safe_convert,IntegerType())

In [0]:
df = (
    df.withColumn(
        "balls_played",
        safe_udf(col("stats_array")[0])
    )
    .withColumn(
        "mintues_batted",
        safe_udf(col("stats_array")[1])
    )
    .withColumn(
        "boundries",
        safe_udf(col("stats_array")[2])
    )
    .withColumn(
        "sixes",
        safe_udf(col("stats_array")[3])
    )
    .withColumn(
        "Strike_rate",
        (when(col("stats_array")[4] == '-',lit("0")).otherwise(col("stats_array")[4])).cast(DecimalType(10,2))
    )
)
# df.display()

In [0]:
%skip
df = (
    df.withColumn(
        "balls_played",
        safe_udf(col("stats_array")[0])
    )
    .withColumn(
        "mintues_batted",
        col("stats_array")[1].cast("int")
    )
    .withColumn(
        "boundries",
        col("stats_array")[2].cast("int")
    )
    .withColumn(
        "sixes",
        col("stats_array")[3].cast("int")
    )
    .withColumn(
        "Strike_rate",
        try:
            col("stats_array")[4].cast("double")
        except:
            lit(0)
    )
)
df.display()

## 03 Fix match date column

In [0]:
df = df.withColumn(
    "match_date",
    to_date(trim(col("date")),"MMMM dd yyyy")
)

# df.display()

## 04 Add derived columns
++ is out, is_not_out, is_run_out

In [0]:
df = (
  df.withColumn(
    "wicket_by_player",
    trim(trim(col("wicket_by")))
  )
  .withColumn(
    "is_out",
    when((col("wicket_by_player")!="-") & (col("wicket_by_player").isNotNull()),lit(1)).otherwise(lit(0))
  )
  .withColumn(
    "is_not_out",
    when(col("wicket_by_player") == '-',lit(1)).otherwise(lit(0))
  )
  .withColumn(
    "is_run_out",
    when(col("wicket_by_player").contains("run out"),lit(1)).otherwise(lit(0))
  )

)

# df.display()

## 05 fix the columns with leading and trailing white spaces

In [0]:
## cols to be corrects

cols = ["id","venue","winner","batting_team","bowling_team","toss","player_name"]

In [0]:
df = df.select(
    *[trim(col(c)).alias(c) if c in cols else col(c) for c in df.columns]
)

## 06 convert runs scored to int

In [0]:
df = df.withColumn(
    "runs_scored",
    col("runs_scored").cast("int")
)

# df.display()

## Match id

In [0]:
df = (
  df.withColumn(
    "season",
    year(col("match_date"))
  )
)
# df.display()

In [0]:
%skip
df.display()

In [0]:
## create final table

df_fin = df.select(
 'id',
 'player_name',
 'balls_played',
 'runs_scored',
 'mintues_batted',
 'boundries',
 'sixes',
 'Strike_rate',
 'wicket_by_player',
 'is_captain',
 'is_out',
 'is_not_out',
 'is_run_out',
 'season'
)

## ++ match uuid

In [0]:
df_fin1 = df_fin.alias("a").join(
  spark.table(f"ipl_database.silver.match_details").alias("b"),
  on = "id",
  how = "inner"
).drop(col("id")).select("a.*","b.match_UUID")

In [0]:
%skip
df_fin1.display()

## Insert

In [0]:
try:
    if spark.catalog.tableExists("ipl_database.silver.batsman_stats"):
        df_fin1.createOrReplaceTempView("df_fin1")
        spark.sql('''
                merge into ipl_database.silver.batsman_stats a
                using df_fin1 b
                on a.match_UUID=b.match_UUID and a.player_name=b.player_name and a.season=b.season
                -- when matched then update set *
                when not matched then insert *
                ''')
        # df_fin.write.format("delta").mode("append").save("abfss://silver@ipldatastorageaccount.dfs.core.windows.net/batting/")
        spark.sql('refresh table ipl_database.silver.batsman_stats')
        print("Data Inserted.")
    else:
        df_fin1.write.format("delta").mode("overwrite").saveAsTable("ipl_database.silver.batsman_stats")
        # spark.sql(f"CREATE TABLE IF NOT EXISTS ipl_database.silver.batsman_stats USING DELTA LOCATION 'abfss://silver@ipldatastorageaccount.dfs.core.windows.net/batting/'")
        print("Table Created.")
except Exception as e:
    print(f"Error while write/append,{e}")